In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.metrics import (classification_report, accuracy_score, confusion_matrix)

## Importing dataset from sklearn

In [2]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

x = pd.DataFrame(housing.data, columns=housing.feature_names)

y = housing.target # median house value is $100,000s

## Data Preprocessing

In [4]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB


In [5]:
x.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


## Data Splitting

In [3]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## Data Scaling

In [13]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## Logistic Model

In [34]:
linear_model = LinearRegression()

linear_model.fit(x_train_scaled, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [35]:
linear_prediction = linear_model.predict(x_test_scaled)

mse = mean_squared_error(y_test, linear_prediction)

print('Mean Squared Error', mse)

Mean Squared Error 0.5558915986952444


In [37]:
score = linear_model.score(x_test_scaled, y_test)

print('R-squared Score:', score * 100)

R-squared Score: 57.578770603245076


## Using Pipeline

In [10]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeRegressor())
])

pipeline.fit(x_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2


In [12]:
y_prediction = pipeline.predict(x_test)

mse = mean_squared_error(y_test, y_prediction)

print('Mean Squared Error:', mse)

Mean Squared Error: 0.49174729730552325


In [15]:
model = pipeline.score(x_test, y_test)

print('Pipeline Score:', model * 100)

Pipeline Score: 62.473753959954735


## Create a Predict Function

In [26]:
def predict_house_price(MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude):

    sample = pd.DataFrame({
        'MedInc' : [MedInc],
        'HouseAge': [HouseAge],
        'AveRooms': [AveRooms],
        'AveBedrms': [AveBedrms],
        'Population': [Population],
        'AveOccup': [AveOccup],
        'Latitude': [Latitude],
        'Longitude':[Longitude]
    })
    prediction = pipeline.predict(sample)[0]

    return f"Estimated House Price: ${prediction * 100_000:,.2f}"

In [27]:
predict_house_price(MedInc=8.4, HouseAge=42.5, AveRooms=7.0, AveBedrms=2.0, 
                    Population=1000, AveOccup=3.5, Latitude=38.5, Longitude=-123.3)

'Estimated House Price: $281,900.00'